# 1. Setup

### Import Statements

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
import matplotlib.pyplot as plt
import pyodbc
import os
import urllib

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

In [2]:
pd.set_option('display.float_format', '{:,.2f}'.format)

### Loading Data

In [16]:
# Config
DB_CONFIG = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=saas_pipeline_db;"
    "Trusted_Connection=yes;"
)

VIEWS = ['v_OBT']

PARQUET_DIR = Path('../data/processed')

# Database Connection
def get_engine():
    conn_str = f"mssql+pyodbc:///?odbc_connect={DB_CONFIG}"
    return create_engine(conn_str, fast_executemany=True)

def test_connection(engine) -> bool:
    try:
        with engine.connect() as conn:
            conn.execute(text("SELECT 1"))
        return True
    except Exception:
        return False

# Load Data
def load_from_sql(engine) -> dict[str, pd.DataFrame]:
    print("Connected to SQL Server, loading data...")
    return {view: pd.read_sql(f"SELECT * FROM {view}", engine) for view in VIEWS}

def load_from_parquet() -> dict[str, pd.DataFrame]:
    print("No Database connection, loading data from parquet...")
    missing = [v for v in VIEWS if not (PARQUET_DIR / f"{v}.parquet").exists()]
    if missing:
        raise FileNotFoundError(
            f"No Database connection and no parquet files for: {', '.join(missing)}\n"
            f"Check Database connection or create parquet files."
        )
    return {view: pd.read_parquet(PARQUET_DIR / f"{view}.parquet") for view in VIEWS}

def load_data() -> dict[str, pd.DataFrame]:
    engine = get_engine()
    # if test_connection(engine):
    conn=False
    if conn:
        data = load_from_sql(engine)
        print(f"Loaded {len(data)} tables from Database.")
    else:
        data = load_from_parquet()
        print(f"Loaded {len(data)} tables from parquet files.")
    return data

In [17]:
data = load_data()
df = data['v_OBT']

No Database connection, loading data from parquet...
Loaded 1 tables from parquet files.


In [18]:
# set number format to float for better number formatting
df['DealValueEUR'] = df['DealValueEUR'].astype(float)
df['AnnualRevenueEUR'] =  df['AnnualRevenueEUR'].astype(float)

In [19]:
df_closed = df[df['Status'] != 'Open'].copy()
df_closed.sample(5)

,DealID,Status,StageName,StageOrder,DealHealthStatus,CreatedDate,LastActivityDate,DealValueEUR,WeightedDealValue,BaseWinProbability,...,LeaveYear,TenureYears,ProductCategory,TotalActivities,TotalDuration,Emails,Calls,Meetings,Demos,Workshops
1464,12694,Won,Contracting,7,Closed,2025-04-16,2025-07-03,"64,761.00","29,142.45",0.05,...,NaN,7.00,CRM,24,1262,9,5,6,2,2
2745,11658,Lost,Qualification,2,Closed,2024-10-23,2025-01-06,"128,019.00","2,560.38",0.11,...,NaN,3.00,BI Reporting,5,156,1,3,1,0,0
1626,12877,Lost,Demo,4,Closed,2025-05-12,2025-07-07,"13,251.00","4,637.85",0.18,...,NaN,7.00,HR Tech,10,578,1,7,0,1,1
2393,11301,Won,Contracting,7,Closed,2024-08-16,2024-09-25,"40,868.00","18,390.60",0.18,...,NaN,3.00,CRM,19,690,9,5,4,0,1
3972,10769,Lost,Demo,4,Closed,2024-05-15,2024-08-07,"12,202.00","3,172.52",0.29,...,NaN,6.00,CRM,12,304,8,3,0,0,1


### Color palette and global chart settings

In [20]:
BLUE = '#4E79A7'
GREEN = '#94B49B'
RED = '#AB5050'
GRAY = '#B0B0B0'
TITLE_COLOR = '#343541'

layout_updates = dict(
    plot_bgcolor='white',
    paper_bgcolor='white',
    title_font=dict(size=20, color=TITLE_COLOR, family='Arial, sans-serif', weight='bold'),
    font=dict(color='#555555'),
    margin=dict(t=80, b=40, l=40, r=40)
)

# 2. Analysis

## Funnel Analysis

In [21]:
funnel_df = pd.DataFrame({
    'StageOrder': range(1,8),
    'StageName': ['Prospecting', 'Qualification', 'Discovery',
                  'Demo', 'Proposal', 'Negotiation', 'Contracting'],
    'DealsEnteredStage': [len(df_closed[df_closed['StageOrder'] >= stage]) for stage in range(1,8)]
})

In [22]:
df_closed['Is_Lost'] = (df_closed['Status'] == 'Lost').astype(int)
loss_funnel_df = df_closed.groupby(['StageOrder', 'StageName']).agg(
    Total_Ended_Here=('DealID', 'count'),
    Lost_Here=('Is_Lost', 'sum')
).reset_index()
loss_funnel_df = loss_funnel_df.sort_values(by='StageOrder', ascending=False)
loss_funnel_df['Deals_Entered_Stage'] = loss_funnel_df['Total_Ended_Here'].cumsum()
loss_funnel_df = loss_funnel_df.sort_values(by='StageOrder', ascending=True).reset_index(drop=True)
loss_funnel_df['Drop_off_Rate_%'] = (loss_funnel_df['Lost_Here'] / loss_funnel_df['Deals_Entered_Stage'] * 100).round(2)
loss_funnel_df = loss_funnel_df.drop(columns=['Total_Ended_Here'])

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Volume (Surviving Deals)', 'Drop-Off Rate (Lost at This Stage)'),
    horizontal_spacing=0.1,
)

fig.add_trace(go.Funnel(
    y=funnel_df['StageName'],
    x=funnel_df['DealsEnteredStage'],
    textinfo='value+percent initial',
    marker=dict(color=BLUE),
    showlegend=False,
    cliponaxis=False
), row=1, col=1)

fig.add_trace(go.Bar(
    y=loss_funnel_df['StageName'],
    x=loss_funnel_df['Drop_off_Rate_%'],
    orientation='h',
    text=loss_funnel_df['Drop_off_Rate_%'].astype(str) + '%',
    textposition='outside',
    marker=dict(color=BLUE, line=dict(color=BLUE, width=0)),
    showlegend=False,
), row=1, col=2)

max_dropoff = loss_funnel_df['Drop_off_Rate_%'].max()
fig.update_yaxes(autorange='reversed', row=1, col=2)
fig.update_xaxes(
    showgrid=False,
    showticklabels=False,
    row=1, col=2,
    range=[0, max_dropoff * 1.2]
)

fig.update_layout(**layout_updates)
fig.update_layout(
    title=dict(
        text=f'1 in 3 Deals Reaches Contracting — Most Lost at Prospecting',
        font=dict(color=TITLE_COLOR, size=22),
        x=0.5,
        xanchor='center'
    ),
)

fig.show()
# fig.write_image("../images/charts/funnel.png", width=1200, height=500, scale=2)

## Seasonality Analysis

In [28]:
df_won = df[df['Status'] == 'Won']

df_won['LastActivityDate'] = pd.to_datetime(df_won['LastActivityDate'])
df_won['Year'] = df_won['LastActivityDate'].dt.year
df_won['Month_Num'] = df_won['LastActivityDate'].dt.month
df_won['Month_Name'] = df_won['LastActivityDate'].dt.strftime('%b')

df_yoy = df_won[df_won['Year'].isin([2024, 2025])]

df_trend = df_yoy.groupby(['Year', 'Month_Num', 'Month_Name'])['DealValueEUR'].sum().reset_index()
df_trend = df_trend.sort_values(['Year', 'Month_Num'])

In [29]:
df_2024 = df_trend[df_trend['Year'] == 2024]
df_2025 = df_trend[df_trend['Year'] == 2025]

last_month_2024 = df_2024['Month_Name'].iloc[-1]
last_val_2024 = df_2024['DealValueEUR'].iloc[-1]

last_month_2025 = df_2025['Month_Name'].iloc[-1]
last_val_2025 = df_2025['DealValueEUR'].iloc[-1]

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_2025['Month_Name'],
    y=df_2025['DealValueEUR'],
    mode='lines+text',
    name='2025 Revenue',
    line=dict(color=BLUE, width=4),
    showlegend=False,
))

fig.add_trace(go.Scatter(
    x=df_2024['Month_Name'],
    y=df_2024['DealValueEUR'],
    mode='lines',
    name='2024 Revenue',
    line=dict(color=GRAY, width=3),
    showlegend=False,
))

fig.update_layout(
    title=dict(
        text='Q4 Drives Disproportionate Revenue in Both Years',
        x=0.5
    )
)
fig.update_layout(**layout_updates)

fig.update_xaxes(
    showgrid=False,
    zeroline=False,
)
fig.update_yaxes(
    title='Revenue (EUR)',
    gridcolor='lightgray',
    zeroline=False
)

fig.add_annotation(
    x=last_month_2024,
    y=last_val_2024,
    text='2024',
    showarrow=False,
    xanchor='left',
    xshift=10,
    font=dict(color=GRAY, size=15, weight='bold', family='Arial, sans-serif'),
)
fig.add_annotation(
    x=last_month_2025,
    y=last_val_2025,
    text='2025',
    showarrow=False,
    xanchor='left',
    xshift=10,
    font=dict(color=BLUE, size=15, weight='bold', family='Arial, sans-serif'),
)

fig.show()
fig.write_image("../images/charts/seasonality.png", width=1100, height=450, scale=2)

## Time in Pipeline (Won/Lost)

In [ ]:
df_won = df[df['Status'] == 'Won']
df_lost = df[df['Status'] == 'Lost']

max_days = df[df['Status'].isin(['Won', 'Lost'])]['DaysInPipeline'].max()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Won Deals', 'Lost Deals'),
    horizontal_spacing=0.08
)

fig.add_trace(go.Histogram(
    x=df_won['DaysInPipeline'],
    histnorm='percent',
    marker_color=GREEN,
    name='Won',
    xbins=dict(size=14),
    marker=dict(line=dict(width=1, color='white')),
    showlegend=False,
), row=1, col=1)

fig.add_trace(go.Histogram(
    x=df_lost['DaysInPipeline'],
    histnorm='percent',
    marker_color=RED,
    name='Lost',
    xbins=dict(size=14),
    marker=dict(line=dict(width=1, color='white')),
    showlegend=False,
), row=1, col=2)

fig.update_layout(**layout_updates)
fig.update_layout(
    title=dict(
        text='Lost Deals Resolve Faster — Won Deals Require More Time',
        x=0.5,
        xanchor='center'
    )
)

fig.update_xaxes(title_text='Days in Pipeline', range=[0, max_days*0.8], showgrid=True, gridcolor='#F0F0F0', row=1, col=1)
fig.update_xaxes(title_text='Days in Pipeline', range=[0, max_days*0.8], showgrid=True, gridcolor='#F0F0F0', row=1, col=2)
fig.update_yaxes(title_text='Share of Deals (%)', showgrid=True, ticksuffix='%', gridcolor='#F0F0F0', row=1, col=1)
fig.update_yaxes(showgrid=True, ticksuffix='%', matches='y', gridcolor='#F0F0F0', row=1, col=2)

fig.show()
# fig.write_image("../images/charts/time-in-pipeline.png", width=1100, height=450, scale=2)

## Ideal Customer Profile

In [35]:
industry_df_pivot = df_closed.pivot_table(
    index='Industry',
    columns='CompanySize',
    values='Status',
    aggfunc=lambda x: (x=='Won').sum() / len(x) * 100
)
industry_df_pivot

CompanySize,Enterprise,Mid-Market,SMB
Industry,,,
Finance,37.06,33.75,32.97
Healthcare,32.30,35.16,39.79
Manufacturing,34.07,29.95,31.38
Retail,34.29,35.83,35.31
Tech,32.50,32.13,34.45


In [ ]:
fig = go.Figure(go.Heatmap(
    z=industry_df_pivot.values,
    x=industry_df_pivot.columns,
    y=industry_df_pivot.index,
    colorscale=[[0, '#F4F7F5'], [1, GREEN]],
    text=industry_df_pivot.values,
    texttemplate="%{text:.1f}%",
    textfont=dict(size=14, color=TITLE_COLOR),
    showscale=True,
))

fig.update_layout(**layout_updates)
fig.update_layout(
    title=dict(
        text='Healthcare SMB and Finance Enterprise Are Easiest to Close',
        x=0.5,
        xanchor='center'
    )
)
fig.update_xaxes(autorange="reversed")

fig.show()
# fig.write_image("../images/charts/ideal-customer-profile.png", width=800, height=450, scale=2)

## Win Rate by Activity Combination

In [37]:
advanced_stages = ['Demo', 'Proposal', 'Negotiation', 'Contracting']
df_advanced = df_closed[df_closed['StageName'].isin(advanced_stages)].copy()

has_demo = df_advanced['Demos'] > 0
has_workshop = df_advanced['Workshops'] > 0

conditions = [
    (~has_demo) & (~has_workshop),
    (has_demo)  & (~has_workshop),
    (~has_demo) & (has_workshop),
    (has_demo)  & (has_workshop)
]

choices = [
    'No Demo, No Workshop',
    'Demo Only',
    'Workshop Only',
    'Demo + Workshop'
]

df_advanced['EngagementLevel'] = np.select(conditions, choices, default='Error')

combo_stats = df_advanced.groupby('EngagementLevel').agg(
    WinRatePct=('Status', lambda x: (x == 'Won').mean() * 100),
    Volume=('Status', 'count')
).reset_index()
combo_stats['WinRatePct'] = combo_stats['WinRatePct'].round(1)

cat_order = ['No Demo, No Workshop', 'Demo Only', 'Workshop Only', 'Demo + Workshop']
combo_stats['EngagementLevel'] = pd.Categorical(combo_stats['EngagementLevel'], categories=cat_order, ordered=True)
combo_stats = combo_stats.sort_values('WinRatePct')

In [39]:
fig = go.Figure(go.Bar(
    x=combo_stats['WinRatePct'],
    y=combo_stats['EngagementLevel'],
    orientation='h',
    marker=dict(color=BLUE, line=dict(width=0)),
    text=combo_stats['WinRatePct'].astype(str) + '%<br><span style="font-size: 11px">(n=' + combo_stats['Volume'].astype(str) + ')</span>',
    textposition='outside',
))

fig.update_layout(
    title=dict(
        text='Each Engagement Activity Adds ~10pp to Win Rate',
        x=0.5,
        xanchor='center'
    )
)

max_winrate = combo_stats['WinRatePct'].max()
fig.update_xaxes(
    title='Win Rate',
    ticksuffix='%',
    gridcolor='#F0F0F0',
    zeroline=False,
    range=[0, max_winrate * 1.1],
)

fig.update_layout(**layout_updates)
fig.show()
fig.write_image("../images/charts/activity-combination.png", width=900, height=400, scale=2)